# 🎓 NLP Mini Project: Practical Benchmarks & Empirical Evaluation
## Title: Agentic RAG-Based AI Research Assistant

**Course:** Natural Language Processing (NLP)  
**Authors / Team Members:**
- **Sameer Sirsath** (PRN: `202401110023`)
- **Rameshwar Sanap** (PRN: `202401110021`)  
**Academic Term:** 2026  

---
### Overview of Syllabus Mapping
| Practical | Topic | Core Techniques | Key Artifacts |
| :--- | :--- | :--- | :--- |
| **Practical 01** | Text Extraction & Preprocessing | PyMuPDF, NLTK Tokenization, Stopword Pruning, Porter Stemmer | `outputs/preprocessing_analysis.txt` |
| **Practical 02** | Sparse vs Dense Vectorization | Scikit-Learn TF-IDF, SentenceTransformers MiniLM, Cosine Sim | `outputs/tfidf_matrix.pkl` |
| **Practical 03** | Sentiment & Rhetorical Tone | DistilBERT fine-tuned on SST-2, Stance Classification | `outputs/sentiment_results.csv` |
| **Practical 04** | Named Entity Recognition | spaCy `en_core_web_sm`, Entity Extraction & Frequency Analysis | `outputs/ner_results.json` |
| **Practical 05** | Topic Modeling | Gensim LDA, Collapsed Gibbs Sampling, Dirichlet Distributions | `outputs/topic_model_results.txt` |
| **Practical 06** | Vector DB & Agentic RAG | Persistent ChromaDB, Hybrid Retrieval, Cyclic LangGraph Swarm | End-to-End Synthesized Brief |

## 1. Setup & Environment Initialization

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
sys.path.append(os.path.abspath(".."))

# Verify active directories
print("Project Root Path:", os.path.abspath(".."))
print("Corpus exists:", os.path.exists("../data/sample_book.pdf"))

## 2. Practical 01: Text Extraction & Morphological Preprocessing

In [2]:
from src.preprocessing import extract_text_from_pdf, run_morphological_benchmarks

# 1. Extract raw pages
pages = extract_text_from_pdf("../data/sample_book.pdf")
print(f"Extracted {len(pages)} pages from academic PDF corpus.")
print("Page 1 Sample Preview:\n", pages[0]["tagged_text"][:350], "...")

# 2. Run Morphological Benchmarks
bench = run_morphological_benchmarks(
    pdf_path="../data/sample_book.pdf",
    output_file="../outputs/preprocessing_analysis.txt"
)

print("\n--- Morphological Reduction Results ---")
print(f"Raw Characters:    {bench['raw_char_count']} -> Cleaned: {bench['clean_char_count']}")
print(f"Raw Tokens:        {bench['raw_token_count']} -> Cleaned: {bench['clean_token_count']} ({bench['token_reduction_pct']:.2f}% reduction)")
print(f"Unique Vocabulary: {bench['raw_vocab_size']} -> Cleaned: {bench['clean_vocab_size']} ({bench['vocab_reduction_pct']:.2f}% reduction)")

print("\nSample Stemming Transformations:")
for item in bench["sample_transformations"][:6]:
    print(" ", item)

## 3. Practical 02: Sparse (TF-IDF) vs Dense (Sentence-BERT) Vectorization

In [3]:
from src.preprocessing import build_tfidf_and_dense_models

vec_models = build_tfidf_and_dense_models(
    pages=pages,
    matrix_path="../outputs/tfidf_matrix.pkl"
)

print(f"TF-IDF Matrix Shape: {vec_models['tfidf_matrix_shape']} (Documents x Features)")
print(f"Dense SBERT Embeddings Shape: {vec_models['dense_embeddings_shape']} (Documents x Hidden Dims)")

# Compare Pairwise Cosine Similarity: Chapter 7 (Transformers) vs Chapter 8 (BERT)
p_trans = 6  # 0-indexed Page 7
p_bert = 7   # 0-indexed Page 8

tfidf_sim = vec_models["tfidf_cosine_sim"][p_trans][p_bert]
dense_sim = vec_models["dense_cosine_sim"][p_trans][p_bert]

print(f"\nCosine Similarity between Page 7 (Transformers) and Page 8 (BERT):")
print(f"- Sparse TF-IDF Cosine Similarity : {tfidf_sim:.4f}")
print(f"- Dense MiniLM Cosine Similarity   : {dense_sim:.4f}")

## 4. Practical 03: Rhetorical Stance & Sentiment Analysis (DistilBERT)

In [4]:
from src.nlp_analyzers import analyze_sentiment_batch, analyze_sentiment_single

# Sample research chunks
sample_chunks = [
    {"content": "The Transformer architecture demonstrates state-of-the-art empirical performance and remarkable scalability across large datasets.", "source": "sample_book.pdf", "page": 7},
    {"content": "Recurrent models suffer from critical vanishing gradients and computational bottlenecks that severely restrict long-horizon sequences.", "source": "sample_book.pdf", "page": 6}
]

sentiment_results = analyze_sentiment_batch(sample_chunks, output_csv="../outputs/sentiment_results.csv")
print(pd.DataFrame(sentiment_results)[["chunk_id", "label", "score", "tone", "preview"]])

## 5. Practical 04: Named Entity Recognition (spaCy)

In [5]:
from src.nlp_analyzers import run_ner_corpus_benchmark

ner_summary = run_ner_corpus_benchmark(pages, output_json="../outputs/ner_results.json")
print(f"Total Named Entities Extracted: {ner_summary['total_entities_detected']}")
print("Entity Distribution by Label:", ner_summary["label_distribution"])
print("\nTop Detected Academic Entities:")
for ent, freq in ner_summary["top_entities"][:8]:
    print(f"  - {ent:<25}: {freq} occurrences")

## 6. Practical 05: Topic Modeling with Gensim LDA

In [6]:
from src.nlp_analyzers import train_lda_topic_model

all_page_texts = [p["raw_text"] for p in pages]
lda_results = train_lda_topic_model(
    texts=all_page_texts,
    num_topics=5,
    passes=10,
    output_file="../outputs/topic_model_results.txt"
)

print("LDA Topic Modeling Summary (5 Topics Discovered):")
for t in lda_results["topics"]:
    keywords = ", ".join([f"{w} ({wt:.3f})" for w, wt in t["weighted_keywords"][:5]])
    print(f"Topic #{t['topic_id'] + 1}: {keywords}")

## 7. Practical 06: Vector DB & 4-Agent LangGraph Research Assistant

In [7]:
from src.graph import run_research_workflow, generate_markdown_report

# Test query
query = "Explain the Self-Attention mechanism and multi-head attention in Transformers."
print(f"Executing Multi-Agent Swarm for query: '{query}'...\n")

result_state = run_research_workflow(query, force_arxiv=False)

print("=== Multi-Agent Execution Trace ===")
for step in result_state["execution_trace"]:
    print(f"[{step['timestamp']}] {step['agent']:<18}: {step['action']}")

print("\n=== Synthesized Academic Response ===")
print(result_state["final_answer"][:800], "...\n")

### Conclusion & Summary
All empirical benchmarks for Practicals 01 through 06 have executed successfully. The results validate:
1. Efficient morphological reduction via NLTK (stemming and stopword filtering).
2. High-dimensional sparse TF-IDF vs dense SBERT semantic similarity capture.
3. Accurate DistilBERT rhetorical stance and spaCy named entity extraction.
4. Meaningful latent topic discovery via Gensim LDA.
5. Cyclic multi-agent reasoning, self-correction, and grounded academic citation generation through LangGraph & ChromaDB.